# BERT-QPP$_{cross}$ on TREC DL 2019 / 2020 — via the repo's own scripts

Same goal as `BERTQPP_TREC_DL_colab.ipynb` (correlate predicted QPP scores against actual MAP@50 / nDCG@100 / nDCG@10 for DL19, DL20, and DL19+20 pooled), but wired to call the repo's own scripts wherever they cover the work, instead of reimplementing that logic in the notebook:
- **`create_test_pkl_files.py`** builds the query+first-doc pickle (replaces our own doc-text-lookup code).
- **`test_CE.py`** runs the trained checkpoint and writes `qid<TAB>score` predictions (replaces our own `CrossEncoder.predict` loop).
- **`evaluation.py`** computes Pearson/Kendall correlation between one predicted-scores file and one actual-scores JSON for a single metric (replaces our own per-system correlation code). Its Spearman output is ignored, since we're not using it.

What's still custom, because nothing in the repo covers it:
- **Ground-truth scoring via `pytrec_eval`** (MAP@50/nDCG@100/nDCG@10) — the repo's own `compute_scores.py` uses PyTerrier and different cutoffs, and can't produce these.
- **Reducing each run file to its rank-1 row** before handing it to `create_test_pkl_files.py` — that script overwrites a query's stored doc text on *every* matching row in the run file rather than picking rank-1 itself, so if it were given a full top-100 run it would silently use whichever row happens to be last in the file. Feeding it a pre-reduced, top-1-only run file keeps that overwrite harmless.
- **Overall** (all systems pooled) and **per-query macro-average** (correlate across systems per query, then average) correlation — `evaluation.py` only ever compares one system's predictions to one system's actuals, so cross-system aggregation has no repo equivalent.

Trade-off worth knowing: `create_test_pkl_files.py` reloads the full MS MARCO collection into memory on every invocation. Called once per (year, system) — 16 times — that's 16 redundant full-collection loads (several minutes total), versus the single shared scan the other notebook does. This version is slower but stays faithful to the repo's scripts as-is.

## 1. Mount Drive, clone the repo, install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Riddhi2587/BERTQPP.git /content/BERTQPP
%cd /content/BERTQPP

!pip install -q sentence-transformers pytrec_eval scipy pandas tqdm gdown

## 2. Configuration — edit paths to match your Drive layout

In [ ]:
import os

drive_base_runfiles_19 = "/content/drive/MyDrive/precise-qpp/data/TRECDL19-runfiles"
drive_base_runfiles_20 = "/content/drive/MyDrive/precise-qpp/data/TRECDL20-runfiles"
drive_base_queries_qrels = "/content/drive/MyDrive/precise-qpp/data/TREC-queries-qrels"

MODEL_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1NDZzEpaay0cDumTKDUSMmv99sg9FyHrL"
MODEL_PATH = "/content/drive/MyDrive/precise-qpp/models/tuned_model-ce_bert-base-uncased_e1_b8"

COLLECTION_PATH = "/content/drive/MyDrive/precise-qpp/data/collection.tsv"  # edit me if you already have it cached elsewhere

OUT_DIR = "/content/drive/MyDrive/precise-qpp/results/bertqpp_cross_dl1920_repo_scripts"
RUN1_DIR = f"{OUT_DIR}/run_top1"
PKL_DIR = f"{OUT_DIR}/pkl"
PRED_DIR = f"{OUT_DIR}/predictions"
ACTUAL_DIR = f"{OUT_DIR}/actual"
POOLED_DIR = f"{OUT_DIR}/pooled"
for d in [OUT_DIR, RUN1_DIR, PKL_DIR, PRED_DIR, ACTUAL_DIR, POOLED_DIR]:
    os.makedirs(d, exist_ok=True)

SYSTEM_NAMES = ["BM25", "rm3", "colbert.e2e", "e5", "monot5", "prf_rank_beta05", "splade", "prf_rerank_beta05"]

def run_files_for_year(base_dir, year):
    """Build the fixed list of per-ranker run files for TREC DL `year` ('2019' or '2020')."""
    yy = year[-2:]
    return [
        f"{base_dir}/BM25.{year}.100.res",
        f"{base_dir}/rm3.100.res",
        f"{base_dir}/colbert.e2e.100.res",
        f"{base_dir}/e5_dl_{yy}.100.res",
        f"{base_dir}/monot5.100.res",
        f"{base_dir}/prf_rank_beta05.{year}.100.res",
        f"{base_dir}/splade.100.res",
        f"{base_dir}/prf_rerank_beta05.{year}.100.res",
    ]

def qrels_path(year):
    return f"{drive_base_queries_qrels}/pass_{year}.qrels"

def queries_path(year):
    return f"{drive_base_queries_qrels}/pass_{year}.queries"

YEAR_RUN_DIRS = {"2019": drive_base_runfiles_19, "2020": drive_base_runfiles_20}
SCOPES = {"DL19": ["2019"], "DL20": ["2020"], "DL19+20": ["2019", "2020"]}
METRIC_KEYS = {"MAP@50": "map_cut_50", "nDCG@100": "ndcg_cut_100", "nDCG@10": "ndcg_cut_10"}
METRICS = set(METRIC_KEYS.values())

## 3. Fetch trained model (cached on Drive via `gdown`)

In [ ]:
if not os.path.isdir(MODEL_PATH) or not os.listdir(MODEL_PATH):
    os.makedirs(MODEL_PATH, exist_ok=True)
    !gdown --folder "$MODEL_DRIVE_FOLDER_URL" -O "$MODEL_PATH"
else:
    print(f"[INFO] Using cached model at {MODEL_PATH}")

## 4. Fetch the MS MARCO collection
Needed by `create_test_pkl_files.py --collection`. `msmarco.blob.core.windows.net` now returns 409 (public access disabled), so this uses the still-public `z22.web.core.windows.net` endpoint with a Dropbox mirror fallback, verified against the known MD5 before extracting.

In [ ]:
import subprocess

MSMARCO_URLS = [
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz",
    "https://www.dropbox.com/s/9f54jg2f71ray3b/collectionandqueries.tar.gz?dl=1",
]
MSMARCO_MD5 = "31644046b18952c1386cd4564ba2ae69"

if not os.path.exists(COLLECTION_PATH):
    os.makedirs(os.path.dirname(COLLECTION_PATH), exist_ok=True)
    archive = "/content/collectionandqueries.tar.gz"

    verified = False
    for url in MSMARCO_URLS:
        print(f"[INFO] Trying {url}")
        subprocess.run(["rm", "-f", archive])
        dl = subprocess.run(["bash", "-c", f'wget -q --show-progress -O "{archive}" "{url}"'])
        if dl.returncode != 0:
            print(f"[WARN] Download failed from {url}, trying next source")
            continue
        md5 = subprocess.run(["md5sum", archive], capture_output=True, text=True).stdout.split()[0]
        if md5 != MSMARCO_MD5:
            print(f"[WARN] MD5 mismatch from {url} (got {md5}, expected {MSMARCO_MD5}), trying next source")
            continue
        verified = True
        break

    if not verified:
        raise RuntimeError(
            f"Could not download a valid collectionandqueries.tar.gz from any known source ({MSMARCO_URLS}). "
            f"Download it manually, extract collection.tsv, and place it at {COLLECTION_PATH} on Drive -- "
            "this cell detects the cached file and skips the download on the next run."
        )

    extract = subprocess.run(["bash", "-c",
        f'tar -xzf "{archive}" -C /content collection.tsv && mv /content/collection.tsv "{COLLECTION_PATH}" && rm "{archive}"'])
    if extract.returncode != 0:
        raise RuntimeError("Archive was verified but extracting collection.tsv from it failed.")
else:
    print(f"[INFO] Using cached collection at {COLLECTION_PATH}")

## 5. Per (year, system): reduce to top-1, build pkl, predict, score ground truth
The only custom parsing here is the full 6-column run file (needed for `pytrec_eval` ground truth, which no repo script produces) and reducing it to a top-1-only 3-column tsv (needed to keep `create_test_pkl_files.py`'s overwrite-per-line behavior correct — see the note above). Everything else after that is the repo's own scripts.

In [ ]:
import json
import pytrec_eval
from collections import defaultdict

def parse_run(path):
    """TREC 6-col run file -> (run_scores: {qid: {docid: score}}, top1_doc: {qid: docid}).
    top1 is picked by lowest rank column, not by file order, so it's robust to unsorted runs."""
    run_scores = defaultdict(dict)
    best_rank = {}
    top1 = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            qid, _, docid, rank, score, _ = parts[:6]
            run_scores[qid][docid] = float(score)
            rank = int(rank)
            if qid not in best_rank or rank < best_rank[qid]:
                best_rank[qid] = rank
                top1[qid] = docid
    return dict(run_scores), top1

year_qrels = {year: pytrec_eval.parse_qrel(open(qrels_path(year))) for year in YEAR_RUN_DIRS}

file_paths = {}  # (year, system) -> {"pred": ..., "actual": ...}
for year, base_dir in YEAR_RUN_DIRS.items():
    print(f"=== DL{year[-2:]} ===")
    for system, run_file in zip(SYSTEM_NAMES, run_files_for_year(base_dir, year)):
        if not os.path.exists(run_file) or os.path.getsize(run_file) == 0:
            print(f"  [SKIP] {system}: missing or empty ({run_file})")
            continue

        run_scores, top1 = parse_run(run_file)

        top1_tsv = f"{RUN1_DIR}/{year}_{system}.tsv"
        with open(top1_tsv, "w") as f:
            for qid, docid in top1.items():
                f.write(f"{qid}\t{docid}\t1\n")

        pkl_path = f"{PKL_DIR}/{year}_{system}.pkl"
        !python3 create_test_pkl_files.py \
            --collection "$COLLECTION_PATH" \
            --queries "{queries_path(year)}" \
            --run "{top1_tsv}" \
            --output "{pkl_path}"

        pred_path = f"{PRED_DIR}/{year}_{system}_pred.txt"
        !python3 test_CE.py \
            --pkl "{pkl_path}" \
            --model "$MODEL_PATH" \
            --output "{pred_path}"

        evaluator = pytrec_eval.RelevanceEvaluator(year_qrels[year], METRICS)
        actual = evaluator.evaluate(run_scores)
        actual_path = f"{ACTUAL_DIR}/{year}_{system}_actual.json"
        json.dump(actual, open(actual_path, "w"))

        file_paths[(year, system)] = {"pred": pred_path, "actual": actual_path}
        print(f"  [OK]   {system}: {len(run_scores)} queries")

## 6. Pool DL19+20 files per system
DL19 and DL20 scopes use each year's files directly. DL19+20 needs the two years' predicted-scores and actual-scores files merged into one file each before `evaluation.py` (which only takes a single file pair) can see the pooled set.

In [ ]:
def resolve_scope_paths(scope, years, system):
    available_years = [y for y in years if (y, system) in file_paths]
    if not available_years:
        return None
    if len(available_years) == 1:
        y = available_years[0]
        return file_paths[(y, system)]["pred"], file_paths[(y, system)]["actual"]

    pred_out = f"{POOLED_DIR}/{scope}_{system}_pred.txt"
    with open(pred_out, "w") as out:
        for y in available_years:
            out.write(open(file_paths[(y, system)]["pred"]).read())

    merged_actual = {}
    for y in available_years:
        merged_actual.update(json.load(open(file_paths[(y, system)]["actual"])))
    actual_out = f"{POOLED_DIR}/{scope}_{system}_actual.json"
    json.dump(merged_actual, open(actual_out, "w"))

    return pred_out, actual_out

def load_predictions(path):
    preds = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                preds[parts[0]] = float(parts[1])
    return preds

def load_actual_json(path):
    return json.load(open(path))

## 7. Correlate — per-system via the repo's `evaluation.py`, overall + per-query macro-average computed directly
`evaluation.py` is imported and called for `per_system` (its Spearman output is discarded). `overall` and `per_query_macro_avg` have no repo equivalent, so they're computed here from the same predicted/actual dicts.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, kendalltau
from evaluation import evaluation as repo_evaluation

rows = []
for scope, years in SCOPES.items():
    system_pred, system_actual = {}, {}
    for system in SYSTEM_NAMES:
        resolved = resolve_scope_paths(scope, years, system)
        if resolved is None:
            continue
        pred_path, actual_path = resolved
        system_pred[system] = load_predictions(pred_path)
        system_actual[system] = load_actual_json(actual_path)

        for metric_label, metric_key in METRIC_KEYS.items():
            common = [qid for qid in system_pred[system]
                      if qid in system_actual[system] and metric_key in system_actual[system][qid]]
            if len(common) < 2:
                continue
            result = repo_evaluation(actual_path, pred_path, metric_key)
            rows.append({"scope": scope, "metric": metric_label, "level": "per_system", "system": system,
                         "n": len(common), "pearson": result["Pearson"], "kendall": result["Kendall"]})

    for metric_label, metric_key in METRIC_KEYS.items():
        # --- overall: every (system, query) pair in the scope pooled together ---
        all_p, all_a = [], []
        for system, pred in system_pred.items():
            actual = system_actual[system]
            for qid in pred:
                if qid in actual and metric_key in actual[qid]:
                    all_p.append(pred[qid])
                    all_a.append(actual[qid][metric_key])
        if len(all_p) >= 2:
            rows.append({"scope": scope, "metric": metric_label, "level": "overall", "system": "ALL",
                         "n": len(all_p), "pearson": pearsonr(all_p, all_a)[0], "kendall": kendalltau(all_p, all_a)[0]})

        # --- per_query_macro_avg: correlate across systems for each query, then average across queries ---
        qids = set()
        for pred in system_pred.values():
            qids.update(pred.keys())

        q_pearsons, q_kendalls = [], []
        for qid in qids:
            p = [system_pred[s][qid] for s in system_pred
                 if qid in system_pred[s] and qid in system_actual[s] and metric_key in system_actual[s][qid]]
            a = [system_actual[s][qid][metric_key] for s in system_pred
                 if qid in system_pred[s] and qid in system_actual[s] and metric_key in system_actual[s][qid]]
            if len(p) < 2:
                continue
            pr, kt = pearsonr(p, a)[0], kendalltau(p, a)[0]
            if not (np.isnan(pr) or np.isnan(kt)):
                q_pearsons.append(pr)
                q_kendalls.append(kt)

        if q_pearsons:
            rows.append({"scope": scope, "metric": metric_label, "level": "per_query_macro_avg", "system": "ALL",
                         "n": len(q_pearsons), "pearson": float(np.mean(q_pearsons)), "kendall": float(np.mean(q_kendalls))})

results_df = pd.DataFrame(rows)
results_df

## 8. Save results

In [ ]:
csv_path = f"{OUT_DIR}/correlations.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved correlation table to {csv_path}")

print("\n--- Per-system (Pearson) ---")
display(results_df[results_df.level == "per_system"].pivot_table(index=["system", "scope"], columns="metric", values="pearson"))

print("\n--- Overall & per-query macro-average (Pearson) ---")
display(results_df[results_df.level != "per_system"].pivot_table(index=["level", "scope"], columns="metric", values="pearson"))